In [2]:
from lmul_nn_funcs import lmul_bits
def lmul(a, b):
    return lmul_bits(a, b)
print("Yes")

Yes


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [6]:
class LSTMLayerAblation(nn.Module):
    def __init__(self, input_size, hidden_size, lmul_gates=False, lmul_states=False, M=7):
        super().__init__()
        self.hidden_size = hidden_size
        self.lmul_gates = lmul_gates
        self.lmul_states = lmul_states
        self.M = M
        self.W = nn.Linear(input_size + hidden_size, 4*hidden_size)

    def forward(self, x_t, h_prev, c_prev):
        combined = torch.cat((x_t, h_prev), dim=1)
        if self.lmul_gates:
            W = self.W.weight
            b = self.W.bias
            prod = lmul(combined.unsqueeze(1), W.unsqueeze(0))
            gates = prod.sum(dim=2) + b
        else:
            gates = self.W(combined)

        i, f, g, o = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        if self.lmul_states:
            c_t = lmul(f, c_prev) + lmul(i, g)
            h_t = lmul(o, torch.tanh(c_t))
        else:
            c_t = f * c_prev + i * g
            h_t = o * torch.tanh(c_t)

        return h_t, c_t


class TinyLLM_Ablation(nn.Module):
    def __init__(self, vocab_size, hidden=128, lmul_gates=False, lmul_states=False):
        super().__init__()
        self.hidden_size = hidden
        self.embed = nn.Embedding(vocab_size, hidden)
        self.lstm = LSTMLayerAblation(hidden, hidden, lmul_gates, lmul_states)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        batch, seq_len = x.size()
        h = torch.zeros(batch, self.hidden_size)
        c = torch.zeros(batch, self.hidden_size)
        for t in range(seq_len):
            emb = self.embed(x[:, t])
            h, c = self.lstm(emb, h, c)
        out = self.fc(h)
        return out

In [7]:
import requests

class CharDataset(torch.utils.data.Dataset):
    def __init__(self, text_chunks, seq_len=128, pad_char=" "):
        self.seq_len = seq_len
        self.pad_char = pad_char
        all_chars = "".join(text_chunks) + pad_char
        unique = sorted(list(set(all_chars)))
        self.stoi = {ch:i for i,ch in enumerate(unique)}
        self.itos = {i:ch for ch,i in self.stoi.items()}
        self.vocab_size = len(unique)   
        self.samples = []
        for chunk in text_chunks:
            padded = chunk[:seq_len].ljust(seq_len, pad_char)
            encoded = torch.tensor([self.stoi[ch] for ch in padded])
            self.samples.append(encoded)
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        seq = self.samples[i]
        return seq[:-1], seq[1:]
#shakespeare data
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
raw_text = requests.get(url).text
lines = raw_text.split("\n")
chunks = []
for i in range(0, len(lines), 5):
    chunk = "\n".join(lines[i:i+5])
    chunks.append(chunk)
ds = CharDataset(chunks[:1000])
train_loader = torch.utils.data.DataLoader(ds, batch_size=2, shuffle=True)
print(f"vocab: {ds.vocab_size}, samples: {len(ds)}")

vocab: 61, samples: 1000


In [8]:
@torch.no_grad()
def get_perplexity(model, dataloader):
    model.eval()
    total = 0
    n = 0
    for x, y in dataloader:
        out = model(x)
        loss = F.cross_entropy(out, y[:, -1])
        total += loss.item()
        n += 1
    avg = total / n
    return torch.exp(torch.tensor(avg)).item()

In [9]:
saved_weights = torch.load("tiny_llm_fp32.pth")
tests = [
    ("FP32 baseline", False, False),
    ("LMUL gates only", True, False),
    ("LMUL states only", False, True),
    ("LMUL both", True, True),
]
print("BREAK")
for label, gates, states in tests:
    m = TinyLLM_Ablation(ds.vocab_size, hidden=128, lmul_gates=gates, lmul_states=states)
    m.load_state_dict(saved_weights)
    ppl = get_perplexity(m, train_loader)
    print(f"{label}: {ppl:.2f}")

BREAK
FP32 baseline: 1.58
LMUL gates only: 1.58
LMUL states only: 1.55
LMUL both: 1.54
